# Estudo dos braços

In [2]:
import pandas as pd

df = pd.read_csv("../data/processed/bank_additional_clean.csv", sep=";")

# Target binário
df["y_bin"] = (df["y"] == "yes").astype(int)

# Braço (proxy de oferta): canal de contato
df["arm"] = df["contact"].map({"cellular": 0, "telephone": 1})

print("Shape:", df.shape)
print("\nDistribuição dos braços:")
print(df["arm"].value_counts())
print("\nTaxa de conversão por braço:")
print(df.groupby("arm")["y_bin"].mean())

Shape: (41188, 22)

Distribuição dos braços:
arm
0    26144
1    15044
Name: count, dtype: int64

Taxa de conversão por braço:
arm
0    0.147376
1    0.052313
Name: y_bin, dtype: float64


# Separação do dataset em features (X), target (Y) e canal (arm)

In [3]:
# Features de contexto do cliente (X) — tudo que descreve o cliente/situação,
# sem vazamento do target e sem a coluna usada como braço (contact)
feature_cols = [
    "age", "job", "marital", "education", "default", "housing", "loan",
    "month", "day_of_week", "campaign", "pdays", "previous", "poutcome",
    "emp.var.rate", "cons.price.idx", "cons.conf.idx", "euribor3m", "nr.employed",
]

X = df[feature_cols].copy()
y = df["y_bin"].copy()
arm = df["arm"].copy()

print("X shape:", X.shape)
print("\nTipos das features:")
print(X.dtypes)

X shape: (41188, 18)

Tipos das features:
age                 int64
job                   str
marital               str
education             str
default               str
housing               str
loan                  str
month                 str
day_of_week           str
campaign            int64
pdays               int64
previous            int64
poutcome              str
emp.var.rate      float64
cons.price.idx    float64
cons.conf.idx     float64
euribor3m         float64
nr.employed       float64
dtype: object


# Cria variáveis dummy

In [5]:
categorical_cols = X.select_dtypes(include=["object", "str"]).columns.tolist()
print("Colunas categóricas:", categorical_cols)

X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

print("\nX_encoded shape:", X_encoded.shape)
print("\nNovas features:")
print(X_encoded.columns.tolist())

Colunas categóricas: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'month', 'day_of_week', 'poutcome']

X_encoded shape: (41188, 51)

Novas features:
['age', 'campaign', 'pdays', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'job_blue-collar', 'job_entrepreneur', 'job_housemaid', 'job_management', 'job_retired', 'job_self-employed', 'job_services', 'job_student', 'job_technician', 'job_unemployed', 'job_unknown', 'marital_married', 'marital_single', 'marital_unknown', 'education_basic.6y', 'education_basic.9y', 'education_high.school', 'education_illiterate', 'education_professional.course', 'education_university.degree', 'education_unknown', 'default_unknown', 'default_yes', 'housing_unknown', 'housing_yes', 'loan_unknown', 'loan_yes', 'month_aug', 'month_dec', 'month_jul', 'month_jun', 'month_mar', 'month_may', 'month_nov', 'month_oct', 'month_sep', 'day_of_week_mon', 'day_of_week_thu', 'day_of_week_tue', 'day_of_week_wed',

# Monta framework do treinamento do modelo
## Teste / Treino

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test, arm_train, arm_test = train_test_split(
    X_encoded, y, arm,
    test_size=0.2,
    random_state=42, # random_state: garante que a divisão seja reproduzível
    stratify=y,      # stratify: garante que a proporção de classes no target seja mantida nos conjuntos de treino e teste
)

print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("\nTaxa de conversão - treino:", y_train.mean().round(4))
print("Taxa de conversão - teste:", y_test.mean().round(4))
print("\nDistribuição de braços - treino:")
print(arm_train.value_counts(normalize=True).round(4))
print("\nDistribuição de braços - teste:")
print(arm_test.value_counts(normalize=True).round(4))

# Salvar para a próxima etapa
X_train.to_csv("../data/processed/X_train.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv", index=False)
y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)
arm_train.to_csv("../data/processed/arm_train.csv", index=False)
arm_test.to_csv("../data/processed/arm_test.csv", index=False)

print("\nArquivos salvos em data/processed/")

X_train: (32950, 51) | X_test: (8238, 51)

Taxa de conversão - treino: 0.1127
Taxa de conversão - teste: 0.1126

Distribuição de braços - treino:
arm
0    0.6345
1    0.3655
Name: proportion, dtype: float64

Distribuição de braços - teste:
arm
0    0.6356
1    0.3644
Name: proportion, dtype: float64

Arquivos salvos em data/processed/


### Sobre os resultados acima:
X_train: (32950, 51) | X_test: (8238, 51)
- 51 features sendo 32950 amostras para treino e 8238 para teste

---
Taxa de conversão - treino: 0.1127
Taxa de conversão - teste: 0.1126
- Para y (y_bin) aproximadamente 11% das amostras são de sucesso
  no geral sem considerar o fator determinado para o 'braço',
  tanto para base de teste quando para a base treino

---
Distribuição de braços - treino:
arm
0    0.6345
1    0.3655

Distribuição de braços - teste:
arm
0    0.6356
1    0.3644
- Do total de ligações, tanto para base de teste quando de treino,
  aproximadamente 64% foram via celular (0) e 36% via telefone (1)

---
O balanceamento da proporção entre treino e teste não é ao acaso,
ele é resultado do parametro stratify

---
Há um dataset apenas para o arm para a possibilidade de modelos
diferentes para cada tipo de contato (celular/telefone), porém
um modelo com o arm como feature também será avaliado.